In [13]:
import sys
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
import h5py as h5

import pandas as pd
import numpy as np

sys.path.append('metaqnn')

from training.one_cycle_lr import OneCycleLR

In [2]:
# Run the following as a regular python program in a terminal, doing so in a notebook is problematic
# ! python metaqnn/main.py RML_2021

In [3]:
# Loading in replay database

df = pd.read_csv('metaqnn/learner_logs/replay_database.csv')

print(df.head())

                                                 net  accuracy  best_accuracy  \
0  [I(1024,1), C(8,2,1), P(2,2), FLAT(1024), SM(24)]  0.188963       0.293095   
1  [I(1024,1), C(4,1,1), P(2,2), C(4,2,1), P(2,2)...  0.245291       0.368286   
2  [I(1024,1), C(4,1,1), BN, P(4,2), C(8,2,1), BN...  0.230596       0.355601   
3  [I(1024,1), C(4,1,1), P(4,2), C(2,1,1), GAP(2)...  0.141883       0.215550   
4  [I(1024,1), C(8,2,1), P(4,2), C(8,1,1), GAP(8)...  0.213451       0.340358   

   trainable_parameters  ix_q_value_update  epsilon  time_finished  
0                 98368                  1      1.0   1.740895e+09  
1                   354                  2      1.0   1.740895e+09  
2                   314                  3      1.0   1.740896e+09  
3                   116                  4      1.0   1.740896e+09  
4                   202                  5      1.0   1.740897e+09  


In [4]:
# Time elapsed for random model generation

start_time = df[df['epsilon'] == 1.0]['time_finished'].iloc[0]
end_time = df[df['epsilon'] == 1.0]['time_finished'].iloc[-1]

time_difference = (end_time - start_time) / 3600.0
print(time_difference)

19.220942113200824


In [5]:
# Best performing model and its layers

pd.set_option('display.max_colwidth', -1)
best_model = df[df['accuracy'] == df['accuracy'].max()]['net']
print(best_model)
pd.set_option('display.max_colwidth', 10)

108    [I(1024,1), C(8,2,1), BN, P(4,4), C(8,2,1), BN, P(2,2), C(8,2,1), P(2,2), C(8,2,1), P(4,4), C(4,2,1), P(4,2), C(4,1,1), P(2,2), FLAT(3), SM(24)]
Name: net, dtype: object


In [6]:
# Accuracy and Best Case Accuracy (accuracy at +30 dB SNR) for best-performing model

accuracies = df[df['accuracy'] == df['accuracy'].max()][['accuracy', 'best_accuracy']]
print(accuracies)

     accuracy  best_accuracy
108  0.345479   0.512123    


In [8]:
old_model = tf.keras.models.load_model('metaqnn/trained_models/RML_2021_0109.keras')
print(old_model.summary())
# Get the input of the old model
input_layer = old_model.input

# Get the output of the layer before the last layer
penultimate_layer = old_model.layers[-2].output  # Assuming the last layer is the output layer

# Create a new model that stops before the original output layer
new_model = tf.keras.Model(inputs=input_layer, outputs=penultimate_layer)
# Create new output layer (initialized with random weights)
new_output_layer = tf.keras.layers.Dense(27, kernel_initializer='glorot_uniform', activation='softmax')(new_model.output)

final_model = tf.keras.Model(inputs=new_model.input, outputs=new_output_layer)

final_model.compile(optimizer=Adam(),
                      loss='categorical_crossentropy',  # Or sparse_categorical_crossentropy
                      metrics=['accuracy'])

print(final_model.summary())

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv1d (Conv1D)              (None, 1024, 8)           40        
_________________________________________________________________
batch_normalization (BatchNo (None, 1024, 8)           32        
_________________________________________________________________
average_pooling1d (AveragePo (None, 256, 8)            0         
_________________________________________________________________
conv1d_1 (Conv1D)            (None, 256, 8)            136       
_________________________________________________________________
batch_normalization_1 (Batch (None, 256, 8)            32        
_________________________________________________________________
average_pooling1d_1 (Average (None, 128, 8)            0         
_________________________________________________________________
conv1d_2 (Conv1D)            (None, 128, 8)            1

In [12]:
# Loading 2021 RadioML dataset (without standard scaling the values)
path = '/home/ashwin/datasets/RADIOML_2021_07_INT8.hdf5'
file_handle = h5.File(path,'r+')

X = file_handle['X'][:]
Y = file_handle['Y'][:]
Z = file_handle['Z'][:]

X_train, X_test, Y_train, Y_test, Z_train, Z_test = train_test_split(X, Y, Z, test_size=0.2, random_state=0)
X_val, X_test, Y_val, Y_test, Z_val, Z_test = train_test_split(X_test, Y_test, Z_test, test_size=0.5, random_state=0)

best_snr_indices = np.where(np.any(Z == 30, axis=1))
best_snr_X, best_snr_Y = X[best_snr_indices], Y[best_snr_indices]

(2875392, 1024, 2)


In [ ]:
# Training 

final_model.fit( 
    x=X_train,
    y=Y_train,
    batch_size=2048,
    epochs=100,
    validation_data=(X_val, 
                     Y_val), 
    callbacks=[
        OneCycleLR(
            max_lr=5e-3, end_percentage=0.2, scale_percentage=0.1,
            maximum_momentum=None,
            minimum_momentum=None, verbose=True
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor = 'val_loss',
            patience = 5,
            mode='auto',
            verbose = 1)
    ],
    verbose=2
)

In [15]:
final_model.save('best_model_checkpoints/best_model.keras')

In [23]:
# General accuracy score
score = final_model.evaluate(X_test, Y_test,  verbose=0, batch_size=1024)
print("General Score: ", score[1] * 100, "%")

General Score:  38.17521035671234 %


In [24]:
# Best Case scenario score
best_case_score= final_model.evaluate(best_snr_X, best_snr_Y,  verbose=0, batch_size=1024)
print("Best Case score: ", best_case_score[1] * 100, "%")

Best Case score:  59.668874740600586 %
